Устанавливаем все нужные библиотеки

In [13]:
%pip install python-docx
%pip install langchain-text-splitters
%pip install qdrant-client sentence-transformers
%pip install -U FlagEmbedding
%pip install -U langchain-ollama
%pip install Ragas
%pip install langchain-huggingface


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Все документы которые у нас есть нужно перевести из docx в json формат для дальнейшей работы. Импортируем нужные библиотеки для работы с ос, расширениями файлов и тд,а так же из докс импортируем класс документ для работы с расширением, затем задем путь где у нас лежат документы и как будет называться наш конечный файл json (r нужна для того что бы питон корректно воспринимал \ в пути), теперь нам нужно почистить наши документы (с этим еще можно поработать), я решил заменить упоминание "федерального, закон" во всех вариациях на пустоту,заменить все пробелы,переносы строк и тд на обычный один пробел, затем наша функция возвращает почищеный текст. Теперь создадим функцию извлечения текста doc = Document(file_path) открывает и загружает в память наш файл, затем создаем список куда мы будем складывать наши абзацы,создаем цикл который итеративно перебирает абзацы, берем наш сырой текст абзаца и прогоняем его через очистку, так же смотрим не получилась ли у нас пустая строчка и если все впорядке то добавляем абзац в список, в возврате мы берем все что у нас получилось и склеиваем вставляя между ними перенос строки. Теперь функция для нашего json файла, создаем список который в конце будет нашим файлом, читаем название всех наших файлов которые в нашей папке data, смотрим что бы все было в формате .docx, склеиваем путь к папке и имя файла, отдаем файл нашей функции и получаем готовый текст, !!сохраняем словарь вместе с текстом и ключом ввиде файла из которого был взят текст,это нужно для того что бы LLM в будущем при цитировании могла точно сказать откуда она взяла тот или иной текст. Теперь открываем файл для записи (указывая 0encoding='utf-8' для адекватной работы русского языка), выгружаем сисок словарей в json файл, (if __name__ == "__main__":) для ннашего запуска всей функции 

In [14]:
import os
import json
import re
from docx import Document

DATA_DIR = "./data"
OUTPUT_FILE = "parsed_documents.json"

def clean_text(text):
   
    text = re.sub(r'\(в ред\. Федеральны[хго]+ закон[аов]+.*?\)', '', text)
    
   
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_docx(file_path):
    doc = Document(file_path)
    full_text = []
    
    for para in doc.paragraphs:
        cleaned_para = clean_text(para.text)
        if cleaned_para:
            full_text.append(cleaned_para)
            
    return "\n".join(full_text)

def main():
    documents = []
    
    
    for filename in os.listdir(DATA_DIR):
        if filename.endswith(".docx"):
            file_path = os.path.join(DATA_DIR, filename)
            print(f"Парсинг файла: {filename}...")
            
            text_content = parse_docx(file_path)
            
            documents.append({
                "source": filename,
                "content": text_content
            })
            
    
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(documents, f, ensure_ascii=False, indent=4)
        
    print(f"Готово! Обработано файлов: {len(documents)}. Результат сохранен в {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Парсинг файла: Государственная информационная система мониторинга в сфере межнациональных и межконфессиональных отношений и раннего предупреждения конфликтных ситуаций.docx...
Парсинг файла: Закон Российской Федерации от 21.07.1993 №5485-1.docx...
Парсинг файла: Приказ ФАДН России от 23.10.2015 №70 (ред. от 08.04.2022, с изм. от 22.06.2023).docx...
Парсинг файла: Федеральный закон от 12.01.1996 №7-ФЗ (ред. от 26.02.2024).docx...
Парсинг файла: Федеральный закон от 24.05.1999 №99-ФЗ (ред. от 23.07.2013).docx...
Готово! Обработано файлов: 5. Результат сохранен в parsed_documents.json


Теперь нам нужно проверить все ли нормально у нас перевелось, имопртируем библиотеку для чтения json и для рандомной генерации (взять какой нибудь текст что бы понять нормально ли все там), создаем функцию в которой мы открываем наш файл в режиме чтения,берем текст и превращаем его обратно в список словарей для python, считаем сколько у нас всего получилось документов, затем заводим счетчик для того что бы понимать длину каждого текста и в конце понять сколько у нас получилось символов, заводим цикл с порядковым номером, вытаскиваем сам файл,его текст и количество символов добавляя их в общую копилку, выводим номер документа и количество символов в нем, а так же общее их количество, теперь мы берем рандомный документ, указываем что бы текст брался из середины для большего понимания текста, затем от этой середины мы берем 1000 символом и их уже выводим

In [15]:
import json
import random

INPUT_FILE = "parsed_documents.json"

def check_parsed_data():
    
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        documents = json.load(f)
        
    print(f"ОБЩАЯ СТАТИСТИКА:")
    print(f"Всего документов в базе: {len(documents)}\n")
    
    total_chars = 0
    for i, doc in enumerate(documents):
        source = doc['source']
        text = doc['content']
        chars_count = len(text)
        total_chars += chars_count
        print(f"[{i+1}] {source}")
        print(f"    Длина текста: {chars_count:,} символов")
    
    print(f"\nОбщий объем всей базы: {total_chars:,} символов")
    
    random_doc = random.choice(documents)
    print(f"\nслучайный фрагмент текста из: {random_doc['source']}\n")
    
    start_idx = len(random_doc['content']) // 2
    snippet = random_doc['content'][start_idx : start_idx + 1000]
    
    print(snippet)

check_parsed_data()

ОБЩАЯ СТАТИСТИКА:
Всего документов в базе: 5

[1] Государственная информационная система мониторинга в сфере межнациональных и межконфессиональных отношений и раннего предупреждения конфликтных ситуаций.docx
    Длина текста: 3,250 символов
[2] Закон Российской Федерации от 21.07.1993 №5485-1.docx
    Длина текста: 59,304 символов
[3] Приказ ФАДН России от 23.10.2015 №70 (ред. от 08.04.2022, с изм. от 22.06.2023).docx
    Длина текста: 7,714 символов
[4] Федеральный закон от 12.01.1996 №7-ФЗ (ред. от 26.02.2024).docx
    Длина текста: 199,597 символов
[5] Федеральный закон от 24.05.1999 №99-ФЗ (ред. от 23.07.2013).docx
    Длина текста: 34,263 символов

Общий объем всей базы: 304,128 символов

случайный фрагмент текста из: Федеральный закон от 24.05.1999 №99-ФЗ (ред. от 23.07.2013).docx

имовыгодной кооперации между ними в соответствии с законодательством Российской Федерации и законодательством иностранных государств.
(в ред. Федерального закона от 23.07.2010 N 179-ФЗ)
2. Органы госуд

Для адекватной работы нам нужно разрезать наши документы на чанки, есть множества способов,но из за специфики наших документов я выберу RecursiveCharacterTextSplitter, он по умному режет текст на чанки ищя сначала абзацы, потом строки и предложения, из за того что юридические документы написаны по хорошей структуре,такой метод чанкирования нам идеальо должен подойти. Импортируем данный метод из библиотеки, а так же указываем файл откда брать текст и куда сохранить наши чанки, создаем функцию где открываем наш файл json и превращаем в файл обратно в список словарей, задаем параметры нарезки, не более 1000 символов один чанк, затем 200 символов первого чанка попадут во второй для сохранения смысла при обрезки, указываем как будем считать размер и ставим тру для указа сплитера на каком символе начался чанк (на всякий случай для будущей отладки). Теперь сам процесс нарезки, создаем пустой список куда мы будем складывать все наши чанки,создаем цикл в котором мы берем наш текст и бьем его на строки где каждая строка это чанк около 1000 символов, затем создаем цикл в котором мы для каждого кусочка создаем словарь в котором есть уникальный ключ,указ из какого текста он вщят и сам текст чанка. Записываем наш результат в новый файл и выводим сколько у нас получилось всего чанков

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

INPUT_FILE = "parsed_documents.json"
OUTPUT_FILE = "chunked_documents.json"

def main():
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        documents = json.load(f)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        add_start_index=True,
    )

    all_chunks = []

    for doc in documents:
        source_name = doc["source"]
        text = doc["content"]
        
        chunks = text_splitter.split_text(text)
        
        for i, chunk_text in enumerate(chunks):
            all_chunks.append({
                "chunk_id": f"{source_name}_{i}",
                "source": source_name,
                "text": chunk_text
            })

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=4)

    print(f"Из {len(documents)} документов создано {len(all_chunks)} смысловых чанков.")
    print(f"Результат сохранен в: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Из 5 документов создано 396 смысловых чанков.
Результат сохранен в: chunked_documents.json


Мы можем реализовать гибридный поиск (по схожости смысла вектора и по ключевым словам), наша модель и qdrant предаставляют такую возможность с помощью создания двух типов векторов. Импортируем нужные библиотеки (с трансформером пришлось приделать костыль, что бы не падал код,не совместим с flag_transform), затем создаем нашу коллекцию которая будет ждать сразу по 2 типа вектора, dense (полные на 1024) и sparse (разряженые которые нужны для поиска ключевых слов), с полными векторами близость будут рассчитывать через косинусное расстояние, загружаем модель и открываем наш json файл, затем загружаем чанки с которыми модель возвращает словари с ключами токенами и значениями их весами, бывает так что два слова могут получить одинаковый айдишник и qdrant падает, поэтому мы смотрим на них и берем только тот у которого наибольший вес так как для нас он представляет большую важность, в qdrant все работает на поинтах,мы кладем в эти поинты айдишник, 2 типа вектора и метаданные (текст в нашем случае), загружая все в базу по 50 штук

In [17]:
import transformers.utils.import_utils
if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    transformers.utils.import_utils.is_torch_fx_available = lambda: False

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, PointStruct, SparseVector
from FlagEmbedding import BGEM3FlagModel
from tqdm.notebook import tqdm


client = QdrantClient(host="localhost", port=6333)
COLLECTION_NAME = "legal_hybrid_base"


print("Пересоздаем коллекцию векторов")
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(size=1024, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)


print("Загружаем BGE-M3")
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=False, device='cpu')

with open("chunked_documents.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)


print(f"Начинаем векторизацию и загрузку {len(chunks)} чанков")

points = []
for i, chunk in enumerate(tqdm(chunks, desc="Обработка чанков")):
    text = chunk["text"]
    
    output = model.encode(text, return_dense=True, return_sparse=True, return_colbert_vecs=False)
    
    dense_vec = output['dense_vecs'].tolist()
    lexical_weights = output['lexical_weights']
    
   
    unique_tokens = {}
    for token_str, weight in lexical_weights.items():
        token_id = model.tokenizer.convert_tokens_to_ids(token_str)
        if token_id is not None:
            if token_id in unique_tokens:
                unique_tokens[token_id] = max(unique_tokens[token_id], float(weight))
            else:
                unique_tokens[token_id] = float(weight)
                
    indices = list(unique_tokens.keys())
    values = list(unique_tokens.values())
            
    point = PointStruct(
        id=i,
        vector={
            "dense": dense_vec,
            "sparse": SparseVector(indices=indices, values=values)
        },
        payload={
            "source": chunk["source"],
            "text": text
        }
    )
    points.append(point)

client.upload_points(
    collection_name=COLLECTION_NAME,
    points=points,
    batch_size=50
)

print("Гибридная база создана")

Пересоздаем коллекцию векторов


C:\Users\asus\AppData\Local\Temp\ipykernel_31344\1576712961.py:16: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Загружаем BGE-M3


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Начинаем векторизацию и загрузку 396 чанков


Обработка чанков:   0%|          | 0/396 [00:00<?, ?it/s]

Гибридная база создана


Теперь соберем гибридный поиск, первым делом говорим нашей базе в qdrant пройтись по тексту разбить на токены слов и все перевести в нижний регистр,чтобы мы могли искать по словам, затем пишем функцию поиска по нашему вопросу, модель берет вопрос и достает оба типа вектора которые помогут нам в определении релевантных чанков, в конце мы вытаскиваем массив со смыслом и превращаем его в питонвский список, тут мы тоже можем столкнуться с проблемой в совпадении токенов,точно так же смотрим и если что берем токен с максимальным весом, в конце упаковываем в SparseVector, теперь включаем фильтр который ищет важные зацепки типо цифр, номеров и так далее, если такие зацепки есть то мы говорим модели искать только там где они есть, теперь мы займемся слиянием,с помощью prefetch мы делаем запрос в qdrant который проводит два поиска по смыслу и ключевым словам (оба ограничены фильтром), с помощью rrf мы берем топ 10 из обоих и если в каждом какой то совпал то rrf дает ему большой буст который выбивает его в топ финальной выдачи, так же добавим для безопасности если у нас есть несколько целей заданные через фильтр,но они разнесены по разным страницам то мы выключаем фильтр и ищем только по смыслу. В конце мы задем наш вопрос и количество релевантых чанков в виде ответа, выводим номер, какой это был документ,какой скор совпадения у него и сам текст

In [18]:
from qdrant_client import models

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="text",
    field_schema=models.TextIndexParams(
        type="text",
        tokenizer="word",
        min_token_len=1,
        lowercase=True
    )
)

def ultimate_hybrid_search(query_text, limit=3):
    print(f"\nИщем: '{query_text}'")
    

    output = model.encode(query_text, return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_vec = output['dense_vecs'].tolist()
    
    unique_tokens = {}
    for token_str, weight in output['lexical_weights'].items():
        token_id = model.tokenizer.convert_tokens_to_ids(token_str)
        if token_id is not None:
            unique_tokens[token_id] = max(unique_tokens.get(token_id, 0), float(weight))
            
    query_sparse = models.SparseVector(
        indices=list(unique_tokens.keys()), 
        values=list(unique_tokens.values())
    )


    exact_terms = re.findall(r'\b\d+(?:-\d+)*\b|[А-Яа-яA-Za-z]+-\d+(?:-\d+)*', query_text)
    
    search_filter = None
    if exact_terms:
        print(f"цели: {exact_terms}.")
        
       
        must_conditions = []
        for term in exact_terms:
            must_conditions.append(
                models.FieldCondition(
                    key="text",
                    match=models.MatchText(text=term)
                )
            )
            
        search_filter = models.Filter(must=must_conditions)

  
    prefetches = [
        models.Prefetch(query=dense_vec, using="dense", limit=10, filter=search_filter),
        models.Prefetch(query=query_sparse, using="sparse", limit=10, filter=search_filter),
    ]
    
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetches,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
        with_payload=True
    ).points

    
    if not results and search_filter:
        print("не нашел точного совпадения. Отключаем фильтр и ищем по смыслу")
        prefetches_fallback = [
            models.Prefetch(query=dense_vec, using="dense", limit=10),
            models.Prefetch(query=query_sparse, using="sparse", limit=10),
        ]
        results = client.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=prefetches_fallback,
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=limit,
            with_payload=True
        ).points

    return results


my_question = "что будет за разглашение гос тайны?"
found_chunks = ultimate_hybrid_search(my_question, limit=3)

for i, chunk in enumerate(found_chunks):
    print(f"\nрезультат {i+1}")
    print(f"Документ: {chunk.payload['source']}")
    print(f"Score: {chunk.score:.4f}")
    print(f"Текст: {chunk.payload['text'][:350]}\n")


Ищем: 'что будет за разглашение гос тайны?'

результат 1
Документ: Закон Российской Федерации от 21.07.1993 №5485-1.docx
Score: 0.5000
Текст: о состоянии экологии, здравоохранения, санитарии, демографии, образования, культуры, сельского хозяйства, а также о состоянии преступности;
о привилегиях, компенсациях и социальных гарантиях, предоставляемых государством гражданам, должностным лицам, предприятиям, учреждениям и организациям;
о фактах нарушения прав и свобод человека и гражданина;
о


результат 2
Документ: Закон Российской Федерации от 21.07.1993 №5485-1.docx
Score: 0.5000
Текст: Раздел VIII. КОНТРОЛЬ И НАДЗОР ЗА ОБЕСПЕЧЕНИЕМ
ЗАЩИТЫ ГОСУДАРСТВЕННОЙ ТАЙНЫ
Статья 30. Контроль за обеспечением защиты государственной тайны
Контроль за обеспечением защиты государственной тайны осуществляют Президент Российской Федерации, Правительство Российской Федерации в пределах полномочий, определяемых Конституцией Российской Федерации, феде


результат 3
Документ: Закон Российской Федерации от 21

Нам нужно протестировать наш поиск, для этого мы собираем сет вопросов и документов которые хотим что бы нас поиск выдавал. Возьмем две метрики hit rate (нашелся ли вообще наш документ на какой либо позиции) и MRR (берет 1/позицию документа, то есть чем выше нужный документ тем больше метрика). Заводим счетчики для метрик и количества вопросов, заводим цикл в котором проходимся по каждому вопросу и на каждый вызываем наш гибридный поиск который дает нам топ 3 ответа которые мы сохраняем в result, теперь по метрикам смотрим на ответы, если ожидаемый источник совпаал с тем,что есть в овтете то ставим флажок,что он справился и кидаем в копилку +1, для mrr делим 1 на позицию документа и так же кидаем в копилку (ранг+1 тк может получится деление на 0), прописываем break тк если нашли правильный документ то другое уже не нужно, если ничего не нашли то прописываем это, в конце переводим в проценты hit rate и считаем средний mrr по всем вопросам и выводим все что было



!!На данный момент метрики даже слишком хорошие, в реальности скорее всего они немного упадут

In [19]:
eval_dataset = [
    {
        "question": "Какой максимальный срок может длиться проверка организации в обычных условиях?",
        "expected_source": "Закон Российской Федерации от 21.07.1993 №5485-1.docx"
    },
    {
        "question": "Какую информацию, касающуюся экологии и здравоохранения, закон запрещает засекречивать?",
        "expected_source": "Закон Российской Федерации от 21.07.1993 №5485-1.docx"
    },
    {
        "question": "Что является основанием для прекращения допуска сотрудника к гостайне по статье 23?",
        "expected_source": "Закон Российской Федерации от 21.07.1993 №5485-1.docx"
    },
    {
        "question": "Куда может обратиться иностранная некоммерческая организация для обжалования действий госорганов?",
        "expected_source": "Федеральный закон от 12.01.1996 №7-ФЗ" 
    },
    {
        "question": "Как осуществляется мониторинг в сфере отношений с соотечественниками за рубежом согласно статье 25?",
        "expected_source": "Федеральный закон от 24.05.1999 №99-ФЗ"
    }
]

print("тест поиска (Hit Rate и MRR)\n")

successful_hits = 0
mrr_score = 0.0
total_questions = len(eval_dataset)

for idx, item in enumerate(eval_dataset):
    question = item["question"]
    expected_doc = item["expected_source"]
    
    print(f"[{idx+1}/{total_questions}] Вопрос: {question}")
    

    results = ultimate_hybrid_search(question, limit=3)
    
    hit = False
    for rank, chunk in enumerate(results):
        
        if expected_doc in chunk.payload['source']:
            hit = True
            successful_hits += 1
            mrr_score += 1.0 / (rank + 1)
            print(f" Успех. Правильный документ найден на {rank+1} месте")
            break 
            
    if not hit:
        print(f" Провал. Правильный документ не попал в Топ-3")
        

hit_rate = (successful_hits / total_questions) * 100
mrr_final = mrr_score / total_questions

print(f"\nФИНАЛЬНЫЕ МЕТРИКИ:")
print(f"Всего вопросов: {total_questions}")
print(f"Успешных нахождений: {successful_hits}")
print(f"Hit Rate: {hit_rate:.1f}%")
print(f"MRR : {mrr_final:.3f}")

тест поиска (Hit Rate и MRR)

[1/5] Вопрос: Какой максимальный срок может длиться проверка организации в обычных условиях?

Ищем: 'Какой максимальный срок может длиться проверка организации в обычных условиях?'
 Успех. Правильный документ найден на 1 месте
[2/5] Вопрос: Какую информацию, касающуюся экологии и здравоохранения, закон запрещает засекречивать?

Ищем: 'Какую информацию, касающуюся экологии и здравоохранения, закон запрещает засекречивать?'
 Успех. Правильный документ найден на 1 месте
[3/5] Вопрос: Что является основанием для прекращения допуска сотрудника к гостайне по статье 23?

Ищем: 'Что является основанием для прекращения допуска сотрудника к гостайне по статье 23?'
цели: ['23'].
 Успех. Правильный документ найден на 1 месте
[4/5] Вопрос: Куда может обратиться иностранная некоммерческая организация для обжалования действий госорганов?

Ищем: 'Куда может обратиться иностранная некоммерческая организация для обжалования действий госорганов?'
 Успех. Правильный документ 

Перейдем к нашей llm, импортируем нужные библиотеки для скачивания модели (я выбрал небольшую модель qwen2.5 на 7млрд параметро тк буду запускать все локально), создаем объект llm,скачиваем нашу модель,температуру ставлю на 0 так как нам не нужна фантазия в  юридических деалх,нужны строгие факты, контекстное окно в 2048 токенов нужно чтобы не перегрзуить память, затем вызваем модель и задаем ей вопрос через обращение invoke и добавляем проверку на случай ошибки, затем прописываем состояние то есть как будет действовать наша модель, на входе она будет получать строку вопроса, qdrant будет выдавать список с текстами и ллм будет отдавать строчку ответа

In [20]:
from typing import TypedDict, List
from langchain_ollama import OllamaLLM 

llm = OllamaLLM(
    model="qwen2.5:7b", 
    temperature=0,
    num_ctx=2048 
)

print(" Проверка связи с Qwen")
try:
    test_response = llm.invoke("Напиши ответ на пример: '2+2'.")
    print(f"Ответ модели: {test_response.strip()}\n")
except Exception as e:
    print(f"Ошибка подключения: {e}")

class GraphState(TypedDict):
    question: str       
    documents: List[str] 
    generation: str     
    
print("Структура State создана")

 Проверка связи с Qwen
Ответ модели: 2 + 2 равно 4.

Структура State создана


Теперь нужны создать узлы (по сути функции которые будет выполнять наш агент), создаем узел поиска в котором мы идем в базу qdrant и достать нужные документы основываясь на вопросе, через documents мы вытаскиваем только чистый текст без всего лишнего, если ничего нет то прописываем в контекст о том, что ничего не нашлось чтобы модель могла нам отказать, затем создаем узел с генерацией ответа в котором мы из state берем вопрос и документы которые положили,склеиваем все 3 в один большой текст что бы моделька работала, прописываем системный промт и отправляем все это самой модели которая возвращает ответ

In [21]:
def retrieve_node(state: GraphState):

    "Узел 1: Поиск документов в базе"

    print("[Узел: Поиск] Ищу релевантные законы")
    question = state["question"]
    
    results = ultimate_hybrid_search(question, limit=3)
    

    documents = []
    for i, chunk in enumerate(results):
        text = chunk.payload.get('text', '')
        source = chunk.payload.get('source', f'Фрагмент №{i+1}') 
        documents.append(f"--- ИСТОЧНИК: {source} ---\n{text}")
    

    if not documents:
        documents = ["К сожалению, в базе не найдено релевантных статей закона."]
        
    return {"documents": documents}


def generate_node(state: GraphState):

    "Узел 2: Генерация ответа с помощью ллм"
    
    print("[Узел: Генерация] Пишу юридический ответ")
    question = state["question"]
    documents = state["documents"]
    

    context = "\n\n".join(documents)
    

    prompt = f"""Ты - профессиональный юрист-консультант. 
Твоя задача - ответить на вопрос пользователя, используя ТОЛЬКО предоставленный контекст из законов РФ.
Если в контексте нет ответа на вопрос, честно скажи: "Я не могу ответить на этот вопрос на основе предоставленных законов".
Не придумывай информацию от себя.

ВАЖНОЕ ПРАВИЛО: В конце ответа обязательно напиши заголовок "Источники:" и перечисли названия документов из контекста, на которые ты опирался.

Контекст (выдержки из законов):
{context}

Вопрос пользователя: {question}

Ответ:"""


    response = llm.invoke(prompt)
    

    return {"generation": response}

print("Узлы успешно созданы")

Узлы успешно созданы


Собираем нашего агента полностью, импортируем маяки для старта и конца, в workflow указываем инструкцию как все должно выглядеть, загружаем нашего поисковика и генератор которые превращаем в ноды, теперь делаем ребра, из вопроса (старта) идем в поисковик из него в генератор и заканчиваем. Если все хорошо то сварачиваем в приложение, потом задаем вопрос и отправляем обращение в модельку,а она выдает свой вопрос



на тестовых вопросах модель показала себя довольно хорошо,видно, что ей не хватает некоторх документов что бы быть полноценым помощником,но она отвечает уже на вопросы по документа и самое главное в случае чего сразу говорит о том что не может ответить из нашей базы

In [22]:
from langgraph.graph import StateGraph, START, END


workflow = StateGraph(GraphState)


workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)


workflow.add_edge(START, "retrieve")

workflow.add_edge("retrieve", "generate")

workflow.add_edge("generate", END)


app = workflow.compile()

print("Конвейер LangGraph успешно собран!\n")

user_question = "Расскажи про систему мониторинга"

print(f"Вопрос пользователя: {user_question}\n")


final_state = app.invoke({"question": user_question})


print(f" Ответ ИИ-юриста:\n\n{final_state['generation']}")

Конвейер LangGraph успешно собран!

Вопрос пользователя: Расскажи про систему мониторинга

[Узел: Поиск] Ищу релевантные законы

Ищем: 'Расскажи про систему мониторинга'
[Узел: Генерация] Пишу юридический ответ
 Ответ ИИ-юриста:

Система мониторинга в сфере межнациональных и межконфессиональных отношений и раннего предупреждения конфликтных ситуаций является государственной информационной системой, созданной для обеспечения условий по решению задач, направленных на своевременное выявление конфликтных и предконфликтных ситуаций. Оператором системы выступает Федеральное агентство по делам национальностей (ФАДН) России.

Основные функции и задачи системы включают:

1. Осуществление регулярного сбора оперативной информации о состоянии и динамике социально-экономических и общественно-политических процессов, а также мониторинг и аналитическую обработку различных источников информации по вопросам реализации государственной национальной политики.
2. Проведение мониторинга в информационно-телек

Для проверки модели создадим датасет вопросов и ответов с источниками которые мы сохраним в json файл

In [23]:
gt_list =[
  {
    "question": "Каковы основные цели функционирования Системы мониторинга в сфере межнациональных отношений?",
    "context": "Целью создания и функционирования Системы мониторинга является реализация Стратегии государственной национальной политики... в части: обеспечения условий для решения задач, направленных на своевременное выявление конфликтных и предконфликтных ситуаций... принятия эффективных управленческих решений... обеспечения возможности оперативного реагирования... учета лиц, относящихся к коренным малочисленным народам.",
    "ground_truth": "Система мониторинга создана для реализации Стратегии государственной национальной политики РФ. Её цели включают: своевременное выявление межнациональных и межконфессиональных конфликтов, поддержку принятия управленческих решений, обеспечение оперативного реагирования на конфликты в регионах и муниципалитетах, а также учет лиц из числа коренных малочисленных народов для обеспечения их прав[cite: 1832, 1833, 1834, 1835, 1836].",
    "source": "Государственная информационная система мониторинга..."
  },
  {
    "question": "Какие сведения закон запрещает относить к государственной тайне и засекречивать?",
    "context": "Не подлежат отнесению к государственной тайне и засекречиванию сведения: о чрезвычайных происшествиях и катастрофах, угрожающих безопасности и здоровью граждан... о состоянии экологии, здравоохранения, санитарии, демографии, образования... о привилегиях, компенсациях и социальных гарантиях... о фактах нарушения прав и свобод человека... о размерах золотого запаса и государственных валютных резервах... о состоянии здоровья высших должностных лиц... о фактах нарушения законности органами государственной власти.",
    "ground_truth": "К государственной тайне нельзя относить сведения о ЧП и катастрофах, угрожающих здоровью граждан; о состоянии экологии, здравоохранения, образования, демографии и преступности; о предоставляемых государством привилегиях и социальных гарантиях; о фактах нарушения прав человека или нарушении законности госорганами; о состоянии здоровья высших лиц страны, а также о размерах золотого запаса и валютных резервов РФ[cite: 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955].",
    "source": "Закон РФ от 21.07.1993 №5485-1"
  },
  {
    "question": "Каков максимальный срок засекречивания сведений, составляющих гостайну?",
    "context": "Срок засекречивания сведений, составляющих государственную тайну, не должен превышать 30 лет. В исключительных случаях этот срок может быть продлен по заключению межведомственной комиссии по защите государственной тайны.",
    "ground_truth": "По общему правилу срок засекречивания не может быть более 30 лет. Однако в исключительных случаях он может быть продлен на основании заключения межведомственной комиссии по защите государственной тайны[cite: 2006, 2007].",
    "source": "Закон РФ от 21.07.1993 №5485-1"
  },
  {
    "question": "Какая организация по закону признается некоммерческой?",
    "context": "Некоммерческой организацией является организация, не имеющая извлечение прибыли в качестве основной цели своей деятельности и не распределяющая полученную прибыль между участниками.",
    "ground_truth": "Некоммерческой организацией (НКО) считается такая организация, основной целью которой не является получение прибыли. Кроме того, НКО не имеет права распределять полученную прибыль между своими участниками[cite: 2280].",
    "source": "Федеральный закон от 12.01.1996 №7-ФЗ"
  },
  {
    "question": "Что признается крупной сделкой для автономной некоммерческой организации?",
    "context": "Для целей настоящего Федерального закона крупной сделкой автономной некоммерческой организации признается сделка или несколько взаимосвязанных сделок, связанные с распоряжением денежными средствами, отчуждением иного имущества... при условии, что цена такой сделки либо стоимость отчуждаемого или передаваемого имущества превышает 10 процентов балансовой стоимости активов... если учредительными документами... не предусмотрен меньший размер.",
    "ground_truth": "Крупной сделкой для АНО считается сделка (или несколько взаимосвязанных сделок) по распоряжению деньгами или имуществом, если её цена или стоимость имущества превышает 10% балансовой стоимости активов организации на последнюю отчетную дату. Уставом АНО может быть установлен более низкий порог для признания сделки крупной[cite: 2613].",
    "source": "Федеральный закон от 12.01.1996 №7-ФЗ"
  },
  {
    "question": "Как часто должны проводиться заседания Консультативного совета по делам национально-культурных автономий?",
    "context": "Заседания Совета проводятся не реже двух раз в год. В случае необходимости могут проводиться внеочередные заседания Совета.",
    "ground_truth": "Заседания Консультативного совета должны проходить как минимум дважды в год. При возникновении необходимости допускается проведение внеочередных заседаний[cite: 2233].",
    "source": "Приказ ФАДН России от 23.10.2015 №70"
  },
  {
    "question": "Кто именно признается соотечественником согласно российскому законодательству?",
    "context": "Соотечественниками являются лица, родившиеся в одном государстве, проживающие либо проживавшие в нем и обладающие признаками общности языка, истории, культурного наследия... соотечественниками за рубежом являются граждане Российской Федерации, постоянно проживающие за пределами территории Российской Федерации. Соотечественниками также признаются лица и их потомки... относящиеся, как правило, к народам, исторически проживающим на территории Российской Федерации... лица, состоявшие в гражданстве СССР... выходцы (эмигранты) из Российского государства, Российской республики, РСФСР, СССР и Российской Федерации.",
    "ground_truth": "Соотечественники — это лица, имеющие общие корни, язык, культуру и историю, а также их прямые потомки. К ним относятся граждане РФ, живущие за границей; лица, сделавшие выбор в пользу духовной связи с Россией, чьи предки жили на её территории; бывшие граждане СССР, живущие в бывших союзных республиках; а также эмигранты из России/СССР/РСФСР и их потомки[cite: 3463, 3464, 3465, 3466, 3467].",
    "source": "Федеральный закон от 24.05.1999 №99-ФЗ"
  },
  {
    "question": "Какие права имеют соотечественники в области образования в Российской Федерации?",
    "context": "Органы государственной власти... способствуют получению соотечественниками образования в образовательных организациях и научных организациях в Российской Федерации. Если иное не установлено федеральными законами... соотечественникам, не являющимся гражданами Российской Федерации, предоставляется наравне с гражданами Российской Федерации право на доступ к образованию... при условии представления ими документов... подтверждающих: гражданство СССР... проживание в прошлом на территории... родство по прямой восходящей линии... проживание за рубежом.",
    "ground_truth": "Соотечественники, не имеющие гражданства РФ, имеют право на доступ к образованию наравне с россиянами при наличии документов, подтверждающих их статус (бывшее гражданство СССР, родство с выходцами из РФ/СССР или проживание на этой территории в прошлом). При установлении квот на обучение иностранных граждан интересы соотечественников должны учитываться в обязательном порядке[cite: 3564, 3565, 3566, 3567, 3568, 3569, 3570, 3574].",
    "source": "Федеральный закон от 24.05.1999 №99-ФЗ"
  },
  {
    "question": "В каких формах могут создаваться общины коренных малочисленных народов?",
    "context": "Общинами коренных малочисленных народов Российской Федерации... признаются формы самоорганизации лиц... объединяемых по кровнородственному (семья, род) и (или) территориально-соседскому принципам.",
    "ground_truth": "Общины малочисленных народов могут создаваться как формы самоорганизации, основанные на кровнородственном (семейном или родовом) либо территориально-соседском принципах[cite: 2399].",
    "source": "Федеральный закон от 12.01.1996 №7-ФЗ"
  },
  {
    "question": "Как иностранная НПО может осуществлять деятельность на территории России?",
    "context": "Иностранная некоммерческая неправительственная организация осуществляет свою деятельность на территории Российской Федерации через свои структурные подразделения - отделения, филиалы и представительства.",
    "ground_truth": "Для ведения деятельности в РФ иностранная некоммерческая неправительственная организация должна создать структурные подразделения: отделения, филиалы или представительства[cite: 2295].",
    "source": "Федеральный закон от 12.01.1996 №7-ФЗ"
  }
]


with open('ground_truth_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(gt_list, f, ensure_ascii=False, indent=4)

print("Файл ground_truth_dataset создан")

Файл ground_truth_dataset создан


Нам нужно протестировать нашего агента для этого я выбрал Ragas это фреймворк который оценивает модель которая использует rag по 3 метрикам faithfulness то есть не придумыает ли модель что-то от себя, Answer Relevancy насколько модели получаются по существу и без лишней воды и Answer Correctness совпадает ли сам ответ с эталоном. Для оценки мы импортируем все нужные библиотеки в том числе pandas что бы красиво сохранить результат dataset для работы самой ragas, загружаем нашего поисковика для оценки схожести векторов и переносим на видеокарту, normalize_embeddings тру для перевода всех векторов к длине 1, это ускоряет подсчеты, затем загружаем сами эталоны ответов открывая файл с нашим датасетом, создаем пустой список куда будем кидать наши метрики для вывода в конце, запускаем цикл который проходит по каждому вопросу закидывая его в нашего агента через invoke, заатем ответ и все остальное упаковываем в словарь для оценки, так же пишем небольшой блок который просто пропустит вопрос если его не получилось обработать, конвертируем results для работы с ragas (max_workers=1, timeout=180 прописываем так как без этого моделька кидает по несколько вопросов и видеокарта не успевает обработать каждый из за этого многие вопросы скипаются), затем происходит сам момент оценки когда моделька оценивает насколько ответил наш агент по метрикам, затем выводим результат и сохраняем все в таблицу csv

In [24]:
import pandas as pd
from datasets import Dataset
from tqdm import tqdm

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_huggingface import HuggingFaceEmbeddings


print("Загрузка модели эмбеддингов BGE-M3")
model_name = "BAAI/bge-m3"
model_kwargs = {'device': 'cuda'} 
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)


print("Загрузка ground_truth_dataset.json")
with open('ground_truth_dataset.json', 'r', encoding='utf-8') as f:
    gt_data = json.load(f)


results = []
print(f" Запуск тестирования агента на {len(gt_data)} вопросах")

for entry in tqdm(gt_data):
    question = entry['question']
    
 
    try:
        response = app.invoke({"question": question})
        
        results.append({
            "question": question,
            "answer": response.get('generation', ""),
            "contexts": response.get('documents', []),
            "ground_truth": entry.get('ground_truth', "")
        })
    except Exception as e:
        print(f"Ошибка при обработке вопроса '{question}': {e}")

dataset = Dataset.from_list(results)

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


run_config = RunConfig(max_workers=1, timeout=180)

print("\n Начинаем оценку качества")


score = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,        
        answer_relevancy,     
        answer_correctness    
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config,
    raise_exceptions=False
)


print("ИТОГОВЫЕ МЕТРИКИ:")
print(score)


df_results = score.to_pandas()
df_results.to_csv("rag_evaluation_final_report.csv", index=False, encoding='utf-8-sig')

print("\n Отчет сохранен в 'rag_evaluation_final_report.csv'")

Загрузка модели эмбеддингов BGE-M3


C:\Users\asus\AppData\Local\Temp\ipykernel_31344\3894992378.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
C:\Users\asus\AppData\Local\Temp\ipykernel_31344\3894992378.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
C:\Users\asus\AppData\Local\Temp\ipykernel_31344\3894992378.py:6: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections impor

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Загрузка ground_truth_dataset.json
 Запуск тестирования агента на 10 вопросах


  0%|          | 0/10 [00:00<?, ?it/s]

...

100%|██████████| 10/10 [01:20<00:00,  8.03s/it]
C:\Users\asus\AppData\Local\Temp\ipykernel_31344\3894992378.py:51: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
C:\Users\asus\AppData\Local\Temp\ipykernel_31344\3894992378.py:52: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)



 Начинаем оценку качества


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[0]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to pa

ИТОГОВЫЕ МЕТРИКИ:
{'faithfulness': 0.6857, 'answer_relevancy': 0.5598, 'answer_correctness': 0.4866}

 Отчет сохранен в 'rag_evaluation_final_report.csv'
